# Suraksha - Synthetic UPI Transaction Data Generator

**Generates 120,000 UPI transactions with 9 fraud types and 35+ features**

Person 1's First Task - Start immediately at 6:30 PM

## Fraud Distribution:
- **Legitimate**: 114,000 (95%)
- **Fraud**: 6,000 (5%)
  - Velocity: 900 (15%)
  - Mule Account: 800 (13%)
  - SIM Swap: 700 (12%)
  - Device Takeover: 700 (12%)
  - Beneficiary Manipulation: 600 (10%)
  - Amount Anomaly: 800 (13%)
  - Temporal Anomaly: 700 (12%)
  - Merchant Fraud: 500 (8%)
  - Failed-Then-Success: 300 (5%)

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import uuid
import hashlib
import os

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configuration
TOTAL_TRANSACTIONS = 120000
FRAUD_RATIO = 0.05
NUM_USERS = 10000
NUM_MERCHANTS = 2000

# Fraud type distribution (within 5% fraud transactions)
FRAUD_DISTRIBUTION = {
    'Velocity Fraud': 0.15,
    'Mule Account': 0.13,
    'SIM Swap': 0.12,
    'Device Takeover': 0.12,
    'Beneficiary Manipulation': 0.10,
    'Amount Anomaly': 0.13,
    'Temporal Anomaly': 0.12,
    'Merchant Fraud': 0.08,
    'Failed-Then-Success': 0.05
}

# Reference data
BANKS = ['HDFC', 'ICICI', 'SBI', 'Paytm', 'PhonePe', 'Axis', 'Kotak', 'Yes Bank']
STATES = ['Maharashtra', 'Delhi', 'Karnataka', 'Tamil Nadu', 'Gujarat', 'Uttar Pradesh', 
          'West Bengal', 'Rajasthan', 'Telangana', 'Andhra Pradesh']
MERCHANT_CATEGORIES = ['Food', 'Shopping', 'Fuel', 'Entertainment', 'Education', 
                      'Travel', 'Utilities', 'Healthcare', 'Groceries', 'Electronics']
TXN_TYPES = ['P2P', 'P2M', 'Bill Payment', 'Recharge']
DEVICE_TYPES = ['Android', 'iOS', 'Web']
NETWORK_TYPES = ['4G', '5G', 'WiFi', '3G']
AGE_GROUPS = ['18-25', '26-35', '36-45', '46-55', '56+']

print("✅ Configuration loaded")
print(f"Target: {TOTAL_TRANSACTIONS:,} transactions")
print(f"Fraud ratio: {FRAUD_RATIO*100}%")

In [0]:
def generate_vpa(user_id, bank):
    """Generate VPA like alice@paytm"""
    names = ['alice', 'bob', 'charlie', 'dave', 'eve', 'frank', 'grace', 'henry', 
             'ivy', 'jack', 'kate', 'leo', 'mia', 'noah', 'olivia', 'peter']
    base_name = random.choice(names)
    return f"{base_name}{user_id % 1000}@{bank.lower()}"

def hash_id(value):
    """Create hashed user ID"""
    return hashlib.md5(str(value).encode()).hexdigest()[:16]

print("✅ Helper functions defined")

In [0]:
print("📊 Generating user profiles...")
users = []
for i in range(NUM_USERS):
    users.append({
        'user_id': hash_id(i),
        'bank': random.choice(BANKS),
        'state': random.choice(STATES),
        'age_group': random.choice(AGE_GROUPS),
        'device_id': f"DEV_{hash_id(i)[:8]}",
        'device_type': random.choice(DEVICE_TYPES),
        'sim_age_days': random.randint(30, 1000),
        'avg_txn_amount': random.uniform(500, 5000)
    })
user_df = pd.DataFrame(users)
print(f"✓ Created {len(users):,} user profiles")
user_df.head()

In [0]:
print("📊 Generating merchant profiles...")
merchants = []
for i in range(NUM_MERCHANTS):
    merchants.append({
        'merchant_id': hash_id(f"merchant_{i}"),
        'merchant_vpa': f"merchant{i}@{random.choice(BANKS).lower()}",
        'category': random.choice(MERCHANT_CATEGORIES),
        'is_high_risk': random.random() < 0.1  # 10% high risk
    })
merchant_df = pd.DataFrame(merchants)
print(f"✓ Created {len(merchants):,} merchant profiles")
merchant_df.head()

In [0]:
# Initialize transaction list
transactions = []

# Date range: 2024-01-01 to 2024-12-31
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)
date_range = (end_date - start_date).days

# Calculate transaction counts
legitimate_count = int(TOTAL_TRANSACTIONS * (1 - FRAUD_RATIO))
fraud_counts = {k: int(TOTAL_TRANSACTIONS * FRAUD_RATIO * v) for k, v in FRAUD_DISTRIBUTION.items()}

# Adjust for rounding
total_fraud = sum(fraud_counts.values())
fraud_counts['Velocity Fraud'] += (int(TOTAL_TRANSACTIONS * FRAUD_RATIO) - total_fraud)

print("📊 Transaction Distribution:")
print(f"  • Legitimate: {legitimate_count:,}")
for fraud_type, count in fraud_counts.items():
    print(f"  • {fraud_type}: {count:,}")

In [0]:
print("⚪ Generating legitimate transactions...")
for i in range(legitimate_count):
    user = user_df.sample(1).iloc[0]
    timestamp = start_date + timedelta(
        days=random.randint(0, date_range),
        hours=random.randint(6, 22),  # Normal hours
        minutes=random.randint(0, 59)
    )
    
    txn_type = random.choice(TXN_TYPES)
    
    txn = {
        'txn_id': f"TXN_{uuid.uuid4().hex[:12]}",
        'timestamp': timestamp,
        'sender_id': user['user_id'],
        'sender_vpa': generate_vpa(hash(user['user_id']), user['bank']),
        'sender_bank': user['bank'],
        'sender_state': user['state'],
        'sender_age_group': user['age_group'],
        'device_id': user['device_id'],
        'device_type': user['device_type'],
        'network_type': random.choice(NETWORK_TYPES),
        'txn_type': txn_type,
        'amount_inr': np.random.lognormal(np.log(user['avg_txn_amount']), 0.5),
        'txn_status': 'SUCCESS',
        'mpin_attempts': 0,
        'device_changed_flag': False,
        'sim_change_recent': False,
        'sim_age_days': user['sim_age_days'],
        'location_changed_flag': False,
        'is_fraud': 0,
        'fraud_type': 'Legitimate',
        'fraud_type_id': 0
    }
    
    # Add receiver details
    if txn_type == 'P2M':
        merchant = merchant_df.sample(1).iloc[0]
        txn['receiver_id'] = merchant['merchant_id']
        txn['receiver_vpa'] = merchant['merchant_vpa']
        txn['receiver_bank'] = random.choice(BANKS)
        txn['receiver_state'] = random.choice(STATES)
        txn['receiver_age_group'] = 'NA'
        txn['merchant_category'] = merchant['category']
        txn['high_risk_merchant'] = merchant['is_high_risk']
    else:
        receiver = user_df.sample(1).iloc[0]
        txn['receiver_id'] = receiver['user_id']
        txn['receiver_vpa'] = generate_vpa(hash(receiver['user_id']), receiver['bank'])
        txn['receiver_bank'] = receiver['bank']
        txn['receiver_state'] = receiver['state']
        txn['receiver_age_group'] = receiver['age_group']
        txn['merchant_category'] = None
        txn['high_risk_merchant'] = False
    
    transactions.append(txn)

print(f"✓ Generated {legitimate_count:,} legitimate transactions")

In [0]:
# Generate all fraud transactions
print("\n🔴 Generating fraud transactions...\n")

fraud_type_id = 1
for fraud_type, count in fraud_counts.items():
    print(f"  Generating {fraud_type}: {count:,} transactions...")
    
    for i in range(count):
        user = user_df.sample(1).iloc[0]
        
        # Fraud-specific patterns
        if fraud_type == 'Velocity Fraud':
            timestamp = start_date + timedelta(days=random.randint(0, date_range), hours=random.randint(0, 23), minutes=random.randint(0, 59))
            amount = random.uniform(1000, 10000)
            device_changed, sim_changed, odd_hours, mpin_attempts, location_changed = False, False, False, 0, False
            
        elif fraud_type == 'Mule Account':
            timestamp = start_date + timedelta(days=random.randint(0, date_range), hours=random.randint(0, 23))
            amount = random.choice([5000, 10000, 15000, 20000, 25000])
            device_changed, sim_changed, odd_hours, mpin_attempts, location_changed = False, False, False, 0, False
            
        elif fraud_type == 'SIM Swap':
            timestamp = start_date + timedelta(days=random.randint(0, date_range), hours=random.randint(0, 23))
            amount = random.uniform(5000, 30000)
            device_changed, sim_changed, odd_hours, mpin_attempts, location_changed = True, True, False, random.randint(1, 2), False
            
        elif fraud_type == 'Device Takeover':
            timestamp = start_date + timedelta(days=random.randint(0, date_range), hours=random.randint(0, 23))
            amount = user['avg_txn_amount'] * random.uniform(2.5, 5)
            device_changed, sim_changed, odd_hours, mpin_attempts, location_changed = True, False, False, random.randint(1, 3), True
            
        elif fraud_type == 'Beneficiary Manipulation':
            hour = random.choice([0, 1, 2, 3, 4, 23])
            timestamp = start_date + timedelta(days=random.randint(0, date_range), hours=hour)
            amount = random.choice([5000, 10000, 15000, 20000])
            device_changed, sim_changed, odd_hours, mpin_attempts, location_changed = False, False, True, random.randint(1, 2), False
            
        elif fraud_type == 'Amount Anomaly':
            timestamp = start_date + timedelta(days=random.randint(0, date_range), hours=random.randint(6, 22))
            amount = random.uniform(40000, 50000)
            device_changed, sim_changed, odd_hours, mpin_attempts, location_changed = False, False, False, 0, False
            
        elif fraud_type == 'Temporal Anomaly':
            hour = random.choice([0, 1, 2, 3, 4, 5, 23])
            timestamp = start_date + timedelta(days=random.randint(0, date_range), hours=hour)
            amount = random.uniform(15000, 35000)
            device_changed, sim_changed, odd_hours, mpin_attempts, location_changed = False, False, True, 0, False
            
        elif fraud_type == 'Merchant Fraud':
            timestamp = start_date + timedelta(days=random.randint(0, date_range), hours=random.randint(0, 23))
            amount = random.uniform(10000, 30000)
            device_changed, sim_changed, odd_hours, mpin_attempts, location_changed = False, False, False, 0, False
            
        elif fraud_type == 'Failed-Then-Success':
            timestamp = start_date + timedelta(days=random.randint(0, date_range), hours=random.randint(0, 23))
            amount = random.choice([5000, 10000, 15000, 20000])
            device_changed, sim_changed, odd_hours, mpin_attempts, location_changed = False, False, False, 0, False
        
        txn_type = random.choice(TXN_TYPES)
        
        txn = {
            'txn_id': f"TXN_{uuid.uuid4().hex[:12]}",
            'timestamp': timestamp,
            'sender_id': user['user_id'],
            'sender_vpa': generate_vpa(hash(user['user_id']), user['bank']),
            'sender_bank': user['bank'],
            'sender_state': user['state'],
            'sender_age_group': user['age_group'],
            'device_id': user['device_id'] if not device_changed else f"DEV_{uuid.uuid4().hex[:8]}",
            'device_type': user['device_type'],
            'network_type': random.choice(NETWORK_TYPES),
            'txn_type': txn_type,
            'amount_inr': amount,
            'txn_status': 'SUCCESS',
            'mpin_attempts': mpin_attempts,
            'device_changed_flag': device_changed,
            'sim_change_recent': sim_changed,
            'sim_age_days': random.randint(1, 7) if sim_changed else user['sim_age_days'],
            'location_changed_flag': location_changed,
            'is_fraud': 1,
            'fraud_type': fraud_type,
            'fraud_type_id': fraud_type_id
        }
        
        # Add receiver details
        if txn_type == 'P2M' or fraud_type == 'Merchant Fraud':
            merchant = merchant_df.sample(1).iloc[0]
            if fraud_type == 'Merchant Fraud':
                high_risk_merchants = merchant_df[merchant_df['is_high_risk'] == True]
                if len(high_risk_merchants) > 0:
                    merchant = high_risk_merchants.sample(1).iloc[0]
            
            txn['receiver_id'] = merchant['merchant_id']
            txn['receiver_vpa'] = merchant['merchant_vpa']
            txn['receiver_bank'] = random.choice(BANKS)
            txn['receiver_state'] = random.choice(STATES)
            txn['receiver_age_group'] = 'NA'
            txn['merchant_category'] = merchant['category']
            txn['high_risk_merchant'] = merchant['is_high_risk'] or (fraud_type == 'Merchant Fraud')
        else:
            receiver = user_df.sample(1).iloc[0]
            txn['receiver_id'] = receiver['user_id']
            txn['receiver_vpa'] = generate_vpa(hash(receiver['user_id']), receiver['bank'])
            txn['receiver_bank'] = receiver['bank']
            txn['receiver_state'] = receiver['state']
            txn['receiver_age_group'] = receiver['age_group']
            txn['merchant_category'] = None
            txn['high_risk_merchant'] = False
        
        transactions.append(txn)
    
    fraud_type_id += 1

print(f"\n✓ Total transactions generated: {len(transactions):,}")

In [0]:
# Create DataFrame and sort by timestamp
print("📊 Creating DataFrame and sorting...")
df = pd.DataFrame(transactions)
df = df.sort_values('timestamp').reset_index(drop=True)

# Add temporal features
print("📊 Adding temporal features...")
df['hour_of_day'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])
df['is_odd_hours'] = df['hour_of_day'].isin([23, 0, 1, 2, 3, 4, 5])

# Add derived features
print("📊 Adding derived features...")
df['amount_is_round'] = (df['amount_inr'] % 1000 == 0) | (df['amount_inr'] % 5000 == 0)

# Round amounts for display
df['amount_inr'] = df['amount_inr'].round(2)

print(f"✓ DataFrame created with {len(df):,} rows and {len(df.columns)} columns")
df.head()

In [0]:
# Get current username dynamically for portability
try:
    username = spark.sql("SELECT current_user()").collect()[0][0]
    print(f"Current user: {username}")
except:
    username = os.environ.get('USER', 'default_user')
    print(f"Using username from environment: {username}")

# Construct portable output path
output_path = f'/Workspace/Users/{username}/Suraksha/data/synthetic_upi_txns.csv'
print(f"\n💾 Saving to {output_path}...")

# Ensure directory exists
output_dir = os.path.dirname(output_path)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"✓ Created directory: {output_dir}")

# Save CSV
df.to_csv(output_path, index=False)
print(f"✓ File saved successfully!")
print(f"  File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

In [0]:
# Summary statistics
print("=" * 60)
print("✅ DATA GENERATION COMPLETE!")
print("=" * 60)

print(f"\n📊 Summary Statistics:")
print(f"  Total transactions: {len(df):,}")
print(f"  Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"  Unique users: {df['sender_id'].nunique():,}")
print(f"  Unique receivers: {df['receiver_id'].nunique():,}")

print(f"\n🎯 Fraud Distribution:")
print(df['fraud_type'].value_counts())

print(f"\n💰 Amount Statistics:")
print(df['amount_inr'].describe())

print(f"\n✅ Ready for ingestion into Delta Lake!")
print("   Next: Run advanced_solution/02_delta_ingestion notebook")